In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import classification_report, accuracy_score, f1_score


remove_metadata = ('headers', 'footers', 'quotes')

train_data = fetch_20newsgroups(subset='train', remove=remove_metadata)
test_data = fetch_20newsgroups(subset='test', remove=remove_metadata)

X_train_raw, y_train = train_data.data, train_data.target
X_test_raw, y_test = test_data.data, test_data.target

count_vectorizer = CountVectorizer(stop_words='english', min_df=2, max_df=0.5)
X_train_counts = count_vectorizer.fit_transform(X_train_raw)
X_test_counts = count_vectorizer.transform(X_test_raw)

binary_vectorizer = CountVectorizer(stop_words='english', min_df=2, max_df=0.5, binary=True)
X_train_binary = binary_vectorizer.fit_transform(X_train_raw)
X_test_binary = binary_vectorizer.transform(X_test_raw)

tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.5)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

In [2]:
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train_counts, y_train)
y_pred_mnb = mnb.predict(X_test_counts)

bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_train_binary, y_train)
y_pred_bnb = bnb.predict(X_test_binary)

mnb_tfidf = MultinomialNB(alpha=0.01)
mnb_tfidf.fit(X_train_tfidf, y_train)
y_pred_mnb_tfidf = mnb_tfidf.predict(X_test_tfidf)

In [3]:
results = {
    "Model": ["Bernoulli NB (Binary)", "Multinomial NB (Count)", "Multinomial NB (TF-IDF)"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bnb),
        accuracy_score(y_test, y_pred_mnb),
        accuracy_score(y_test, y_pred_mnb_tfidf)
    ],
    "Macro F1-Score": [
        f1_score(y_test, y_pred_bnb, average='macro'),
        f1_score(y_test, y_pred_mnb, average='macro'),
        f1_score(y_test, y_pred_mnb_tfidf, average='macro')
    ],
    "Weighted F1-Score": [
        f1_score(y_test, y_pred_bnb, average='weighted'),
        f1_score(y_test, y_pred_mnb, average='weighted'),
        f1_score(y_test, y_pred_mnb_tfidf, average='weighted')
    ]
}

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

                  Model  Accuracy  Macro F1-Score  Weighted F1-Score
  Bernoulli NB (Binary)  0.485130        0.467865           0.480839
 Multinomial NB (Count)  0.651089        0.617859           0.629644
Multinomial NB (TF-IDF)  0.694636        0.680073           0.690832
